## Prerequisite Code

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ../initial-setup/03-utils

In [0]:
# Create widgets
dbutils.widgets.text('catalog', 'sportsdirect_sales', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

# Access widgets
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
# Define source directory
source_dir = f's3://sd-warrior-acquisition/orders/landing/*.csv'

# Define target directory
target_dir = f's3://sd-warrior-acquisition/orders/archive'

## Warrior Bronze Layer

In [0]:
# Get the raw data
raw_data = spark.read \
    .format('csv') \
    .option('header', True) \
    .option('inferSchema', True) \
    .load(source_dir) \
    .withColumn('read_timestamp', F.current_timestamp()) \
    .select('*', '_metadata.file_name', '_metadata.file_size')

In [0]:
# View raw data
display(raw_data)

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FDEC83401502,"Tuesday, December 02, 2025",789401,25891203,256.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891502,218.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891403,280.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891201,262.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891203,256.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC83401502,"Tuesday, December 02, 2025",789401,25891403,280.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC84202603,"Tuesday, December 02, 2025",789202,25891502,218.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC84202603,2025/12/02,789202,25891403,267.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC84202603,"Tuesday, December 02, 2025",789202,25891601,69.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621
FDEC84202603,"Tuesday, December 02, 2025",789202,25891602,126.0,2026-03-19T05:38:13.706Z,orders_2025_12_02.csv,19621


In [0]:
# Write raw data to the bronze table
raw_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', True) \
            .mode('append') \
                .saveAsTable(f'{catalog}.{wr_bronze_schema}.fact_order')

In [0]:
# Create a staging table to process incremental data
raw_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', True) \
            .mode('overwrite') \
                .saveAsTable(f'{catalog}.{wr_bronze_schema}.fact_order_staging')

In [0]:
# Move processed files to the archive folder
processed_files = dbutils.fs.ls('s3://sd-warrior-acquisition/orders/landing/')

for file in processed_files:
    if file.name.endswith('.csv'):
        dbutils.fs.mv(file.path, f'{target_dir}/{file.name}', True)

## Warrior Silver Layer

In [0]:
# Clean data and apply transformations

transformed_data = spark.sql(f'SELECT * FROM {catalog}.{wr_bronze_schema}.fact_order_staging')

# Remove the weekday from order_placement_date
transformed_data = transformed_data \
    .withColumn(
        'order_placement_date',
        F.regexp_replace(
            'order_placement_date',
            r'([A-Za-z]+, )?',
            ''
        )
    )

# Convert order_placement_date to date
transformed_data = transformed_data \
    .withColumn(
        'order_placement_date',
        F.coalesce(
            F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
            F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
            F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
            F.try_to_date('order_placement_date', 'MMMM dd, yyyy')
        )
    )

# Rename order_placement_date to date
transformed_data = transformed_data \
    .withColumnRenamed(
        'order_placement_date',
        'date'
    )

# Replace invalid customer_ids with '999999'
transformed_data = transformed_data \
    .withColumn(
        'customer_id',
        F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id'))
        .otherwise('999999')
    )

# Rename customer_id to customer_code
transformed_data = transformed_data \
    .withColumnRenamed(
        'customer_id',
        'customer_code'
    )

# Convert product_id to string
transformed_data = transformed_data \
    .withColumn(
        'product_id',
        F.col('product_id').cast('string')
    )

# Add product codes
transformed_data = transformed_data \
    .join(
        spark.sql(f'SELECT * FROM {catalog}.{wr_silver_schema}.dim_product'),
        on='product_id',
        how='inner'
    ) \
        .select(
            transformed_data['order_id'],
            transformed_data['date'],
            transformed_data['customer_code'],
            transformed_data['product_id'],
            transformed_data['order_qty'],
            transformed_data['read_timestamp'],
            transformed_data['file_name'],
            transformed_data['file_size'],
            F.col('dim_product.product_code').alias('product_code')
        )

# Remove orders with no quantity
transformed_data = transformed_data.filter(F.col('order_qty') > 0)

# Convert order_qty to int
transformed_data = transformed_data \
    .withColumn(
        'order_qty',
        F.col('order_qty').cast('int')
    )

# Rename order_qty to sold_quantity
transformed_data = transformed_data \
    .withColumnRenamed(
        'order_qty',
        'sold_quantity'
    )

# Delete duplicate orders
transformed_data = transformed_data.dropDuplicates(['order_id', 'date', 'customer_code', 'product_code', 'sold_quantity'])

In [0]:
# Verify transformed data
display(transformed_data.orderBy('order_id'))

order_id,date,customer_code,product_id,sold_quantity,read_timestamp,file_name,file_size,product_code
FDEC83101601,2025-12-02,789101,25891601,93,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb
FDEC83101601,2025-12-02,789101,25891303,35,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5
FDEC83102102,2025-12-02,789102,25891102,462,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f
FDEC83102402,2025-12-02,789102,25891402,289,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,41d9f8038c4771bf55fdddeaf9e940f5e23b717d63c6a992e2654afc37fc2c8d
FDEC83103201,2025-12-02,789103,25891103,342,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,6a18f762edaea4192d8a27e46560c29aecf7b2689792172da9a7c1c4b5532909
FDEC83103201,2025-12-02,789103,25891201,318,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,2e73a3ac7f86e06279f1d690401af67a2c6109457d025f9b0f459bd708a60689
FDEC83103201,2025-12-02,789103,25891101,484,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,95dd546ad1c0e319431aabb5d05da6af9d0418dbf7decda178337b8c26cc898f
FDEC83103603,2025-12-02,789103,25891603,112,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,798a750ff46551522957fee953cee201003fd974698c192efb3e4649be4c9652
FDEC83121101,2025-12-02,789121,25891101,407,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,95dd546ad1c0e319431aabb5d05da6af9d0418dbf7decda178337b8c26cc898f
FDEC83121203,2025-12-02,789121,25891203,482,2026-03-19T05:46:40.499Z,orders_2025_12_02.csv,19621,7cde4fee80f465659932d7e9336957d070865e23db86fb63dd73a40dc309f430


In [0]:
# Write transformed data to the silver table
transformed_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', 'true') \
            .option('mergeSchema', 'true') \
                .mode('append') \
                    .saveAsTable(f'{catalog}.{wr_silver_schema}.fact_order')

In [0]:
# Create a staging table to process incremental data
transformed_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', True) \
            .mode('overwrite') \
                .saveAsTable(f'{catalog}.{wr_silver_schema}.fact_order_staging')

## Warrior Gold Layer

In [0]:
# Get the transformed data
analytics_data = spark.sql(f'SELECT * FROM {catalog}.{wr_silver_schema}.fact_order_staging')

In [0]:
# Select necessary columns
analytics_data = analytics_data.select('date', 'product_code', 'customer_code', 'sold_quantity')

In [0]:
# View analytics data and load to the gold table

display(analytics_data)

analytics_data.write \
    .format('delta') \
        .option('enableChangeDataFeed', 'true') \
            .option('mergeSchema', 'true') \
                .mode('append') \
                    .saveAsTable(f'{catalog}.{wr_gold_schema}.fact_order')

date,product_code,customer_code,sold_quantity
2025-12-02,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5,789321,24
2025-12-02,6a18f762edaea4192d8a27e46560c29aecf7b2689792172da9a7c1c4b5532909,789703,469
2025-12-02,7cde4fee80f465659932d7e9336957d070865e23db86fb63dd73a40dc309f430,789103,261
2025-12-02,6a18f762edaea4192d8a27e46560c29aecf7b2689792172da9a7c1c4b5532909,789201,394
2025-12-02,41d9f8038c4771bf55fdddeaf9e940f5e23b717d63c6a992e2654afc37fc2c8d,789721,491
2025-12-02,8395d734ede7e81a35c67ea3ae7240db0380f8370fb488a3872ea033af705ee4,789601,334
2025-12-02,19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb,789101,93
2025-12-02,c3a270caf0285f44bbd624e9a209a46e86f47adf81d2bcadf5969ff260d97d12,999999,203
2025-12-02,2e73a3ac7f86e06279f1d690401af67a2c6109457d025f9b0f459bd708a60689,789321,164
2025-12-02,836744df97fd09ea8a22b4693e616edd9fbda3f34a1262dcf36b75148f80c6ce,789421,165


## Sports Direct Gold Layer

In [0]:
# Merge data from the Warrior gold table to the Sports Direct gold table

# Get the Warrior silver staging table data
fact_order_staging_table = spark.sql(f'SELECT * FROM {catalog}.{wr_silver_schema}.fact_order_staging')

# Get unique months from the staging table
staging_table_months = fact_order_staging_table \
    .select(F.trunc('date', 'MM').alias('date')) \
        .distinct()

# Verify months
display(staging_table_months)

# Create a temporary view
staging_table_months.createOrReplaceTempView('staging_table_months')

date
2025-12-01


In [0]:
# Get the incremental order data
incremental_fact_order_table = spark.sql(
    f'''
        SELECT 
            f.date, 
            f.product_code, 
            f.customer_code, 
            f.sold_quantity
        FROM 
            {catalog}.{wr_gold_schema}.fact_order AS f
        INNER JOIN 
            staging_table_months AS m
            ON 
                trunc(f.date, 'MM') = m.date
    '''
)

In [0]:
# Aggregate sold_quantity by month
incremental_fact_order_table = incremental_fact_order_table \
    .withColumn(
        'date',
        F.trunc('date', 'MM')
    ) \
        .groupBy(
            'date',
            'customer_code',
            'product_code'
        ) \
            .agg(
                F.sum('sold_quantity').alias('sold_quantity')
            )

In [0]:
# Verify incremental_fact_order_table
display(incremental_fact_order_table)

date,customer_code,product_code,sold_quantity
2025-12-01,789622,19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb,255
2025-12-01,789122,19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb,85
2025-12-01,789121,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5,92
2025-12-01,789201,6a18f762edaea4192d8a27e46560c29aecf7b2689792172da9a7c1c4b5532909,809
2025-12-01,789521,95dd546ad1c0e319431aabb5d05da6af9d0418dbf7decda178337b8c26cc898f,972
2025-12-01,789221,81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c,136
2025-12-01,789622,d5f5123af988ee1eb64fa302fdbc78b2965b9d0f9d0e68d51c50bfcaa30d6210,26
2025-12-01,789320,d5f5123af988ee1eb64fa302fdbc78b2965b9d0f9d0e68d51c50bfcaa30d6210,34
2025-12-01,789102,62254dca28e1f3ce45668a4abc8571ad4fc3923df56d7a0c12c369dc76aa1f67,562
2025-12-01,789902,62254dca28e1f3ce45668a4abc8571ad4fc3923df56d7a0c12c369dc76aa1f67,533


In [0]:
# Get the Sports Direct gold table
sd_fact_order_table = DeltaTable.forName(spark, f'{catalog}.{sd_gold_schema}.fact_order')

In [0]:
# Merge data
sd_fact_order_table.alias('target').merge(
    incremental_fact_order_table.alias('source'),
    'target.date = source.date AND target.product_code = source.product_code AND target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql

-- Delete staging tables
DROP TABLE IF EXISTS sportsdirect_sales.warrior_bronze.fact_order_staging;
DROP TABLE IF EXISTS sportsdirect_sales.warrior_silver.fact_order_staging;